In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             mean_absolute_error, mean_squared_error)
import joblib

# Cell 2: Load Model and Test Data
model = tf.keras.models.load_model('forex_analysis_complete.h5')
test_data = np.load('test_data.npy')  # Would need to save from training
y_test = np.load('y_test.npy', allow_pickle=True)

# Or recreate from previous notebook
# For this demo, we'll assume test data is available

print("Model loaded successfully")

# Cell 3: Make Predictions on Test Set
predictions = model.predict(X_test)

# Extract predictions
pred_direction = np.argmax(predictions[0], axis=1)
pred_volatility = np.argmax(predictions[1], axis=1)
pred_price_level = (predictions[2] > 0.5).astype(int).flatten()
pred_trend = predictions[3].flatten()

# Cell 4: Direction Analysis
print("="*50)
print("DIRECTION PREDICTION ANALYSIS")
print("="*50)
dir_accuracy = accuracy_score(y_test[0], pred_direction)
print(f"Accuracy: {dir_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test[0], pred_direction, target_names=['Down', 'Up']))

# Confusion Matrix
cm_dir = confusion_matrix(y_test[0], pred_direction)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_dir, annot=True, fmt='d', cmap='Blues')
plt.title('Direction Prediction Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Cell 5: Volatility Regime Analysis
print("="*50)
print("VOLATILITY REGIME ANALYSIS")
print("="*50)
vol_accuracy = accuracy_score(y_test[1], pred_volatility)
print(f"Accuracy: {vol_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test[1], pred_volatility, target_names=['Low', 'Medium', 'High']))

# Confusion Matrix
cm_vol = confusion_matrix(y_test[1], pred_volatility)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_vol, annot=True, fmt='d', cmap='Greens')
plt.title('Volatility Regime Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Cell 6: Price Level Analysis
print("="*50)
print("PRICE LEVEL ANALYSIS (Support/Resistance)")
print("="*50)
price_accuracy = accuracy_score(y_test[2], pred_price_level)
print(f"Accuracy: {price_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test[2], pred_price_level, target_names=['Normal', 'Near Resistance']))

# Cell 7: Trend Strength Analysis
print("="*50)
print("TREND STRENGTH ANALYSIS")
print("="*50)
trend_mae = mean_absolute_error(y_test[3], pred_trend)
trend_rmse = np.sqrt(mean_squared_error(y_test[3], pred_trend))
print(f"MAE: {trend_mae:.4f}")
print(f"RMSE: {trend_rmse:.4f}")

# Actual vs Predicted plot
plt.figure(figsize=(15, 6))
plt.plot(y_test[3][:200], label='Actual Trend Strength', alpha=0.7)
plt.plot(pred_trend[:200], label='Predicted Trend Strength', alpha=0.7)
plt.title('Trend Strength Prediction (Normalized ADX)')
plt.xlabel('Time Step')
plt.ylabel('Trend Strength')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Cell 8: Backtesting Simulation
def backtest_strategy(df, predictions_direction, initial_capital=10000):
    """
    Simple backtesting based on direction predictions
    """
    capital = initial_capital
    positions = []
    returns = []
    
    for i, pred in enumerate(predictions_direction):
        if i >= len(df) - 1:
            break
            
        if pred == 1:  # Predict Up
            # Buy at close, sell at next close
            daily_return = df['Close'].iloc[i+1] / df['Close'].iloc[i] - 1
        else:  # Predict Down
            daily_return = - (df['Close'].iloc[i+1] / df['Close'].iloc[i] - 1)
        
        capital = capital * (1 + daily_return)
        returns.append(daily_return)
        positions.append(pred)
    
    return capital, returns

# Load original data for backtesting
df_original = pd.read_csv('EURUSD_clean.csv', index_col=0, parse_dates=True)
final_capital, strategy_returns = backtest_strategy(df_original.iloc[len(df_original)-len(pred_direction):], 
                                                    pred_direction)

# Calculate benchmark (buy and hold)
buy_hold_return = df_original['Close'].iloc[-1] / df_original['Close'].iloc[-len(pred_direction)] - 1

print("="*50)
print("BACKTESTING RESULTS")
print("="*50)
print(f"Initial Capital: $10,000")
print(f"Final Capital (AI Strategy): ${final_capital:.2f}")
print(f"Total Return (AI): {(final_capital/10000 - 1)*100:.2f}%")
print(f"Buy & Hold Return: {buy_hold_return*100:.2f}%")
print(f"Sharpe Ratio (AI): {np.mean(strategy_returns)/np.std(strategy_returns)*np.sqrt(252):.3f}")

# Cell 9: Feature Importance Analysis
# Use XGBoost for feature importance
import xgboost as xgb

# Create simple XGB model for feature importance
feature_names = feature_columns[:100]  # Limit for visualization
xgb_model = xgb.XGBClassifier(n_estimators=100, max_depth=5)
xgb_model.fit(X_train.reshape(X_train.shape[0], -1)[:, :100], y_train[0])

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'], importance_df['importance'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importance for Direction Prediction')
plt.gca().invert_yaxis()
plt.show()

# Cell 10: Save Evaluation Results
results = {
    'direction_accuracy': dir_accuracy,
    'volatility_accuracy': vol_accuracy,
    'price_level_accuracy': price_accuracy,
    'trend_mae': trend_mae,
    'backtest_return': (final_capital/10000 - 1),
    'sharpe_ratio': np.mean(strategy_returns)/np.std(strategy_returns)*np.sqrt(252)
}

results_df = pd.DataFrame([results])
results_df.to_csv('evaluation_results.csv', index=False)
print("Evaluation results saved to 'evaluation_results.csv'")
print(results_df)